In [1]:
#Spark connection with S3 options
import os
import socket
from pyspark.sql import SparkSession

# Замените следующие значения на свои credentials
aws_access_key = "j612yzOjcIwYyPhVD14P"
aws_secret_key = "bVTEwJ7sJ0TboubM5wJcn2ieYrV91uXk3N9dJ457"
s3_bucket = "startde-datasets"
s3_endpoint_url = "https://s3.lab.karpov.courses"
 
APACHE_MASTER_IP = socket.gethostbyname("apache-spark-master-0.apache-spark-headless.apache-spark.svc.cluster.local")
APACHE_MASTER_URL = f"spark://{APACHE_MASTER_IP}:7077"
POD_IP = os.environ["MY_POD_IP"]
SPARK_APP_NAME = f"spark-{os.environ['HOSTNAME']}"
JARS = """/nfs/env/lib/python3.8/site-packages/pyspark/jars/clickhouse-native-jdbc-shaded-2.6.5.jar, 
/nfs/env/lib/python3.8/site-packages/pyspark/jars/hadoop-aws-3.3.4.jar,
/nfs/env/lib/python3.8/site-packages/pyspark/jars/aws-java-sdk-bundle-1.12.433.jar,
/nfs/env/lib/python3.8/site-packages/pyspark/jars/postgresql-42.7.4.jar
"""

MEM = "512m"
CORES = 1
 
spark = SparkSession.\
        builder.\
        appName(SPARK_APP_NAME).\
        master(APACHE_MASTER_URL).\
        config("spark.executor.memory", MEM).\
        config("spark.jars", JARS).\
        config("spark.executor.cores", CORES).\
        config("spark.driver.host", POD_IP).\
        config("spark.hadoop.fs.s3a.access.key", aws_access_key). \
        config("spark.hadoop.fs.s3a.secret.key", aws_secret_key). \
        config("fs.s3a.endpoint", s3_endpoint_url).  \
        config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem"). \
        config("spark.hadoop.fs.s3a.committer.name", "directory"). \
        config("spark.hadoop.fs.s3a.path.style.access", True). \
        config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"). \
        config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false"). \
        getOrCreate()

26/02/08 10:39:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Read sales table
sales_table = spark.read.parquet("s3a://startde-datasets/shared/sales.parquet")

26/02/08 10:39:22 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [3]:
sales_table.show()

+--------+----------+---------+----------+---------------+--------------------+
|order_id|product_id|seller_id|      date|num_pieces_sold|       bill_raw_text|
+--------+----------+---------+----------+---------------+--------------------+
|       1|         0|        0|2023-07-02|              7|itgntlqljxnCgkeax...|
|       2|         0|        0|2023-07-05|              9|ymkndxcgbyyqugbgp...|
|       3|         0|        0|2023-07-05|              7|qwbfznfqwfgkuXrnn...|
|       4|         0|        0|2023-07-03|              5|tybsrjpfucpitwqxk...|
|       5|         0|        0|2023-07-04|              3|crrjscajxlmugzmyg...|
|       6|         0|        0|2023-07-03|              1|blxycrmmzmaiarooj...|
|       7|         0|        0|2023-07-09|              8|ymidneomjcfnircqz...|
|       8|         0|        0|2023-07-02|              5|euavsmtstttcjxipl...|
|       9|         0|        0|2023-07-01|              7|jpwsylnygmxdfnswc...|
|      10|         0|        0|2023-07-0

In [7]:
result = sales_table.groupBy(col("date")).agg(sum(col("num_pieces_sold")).alias("total_units_sold")). \
    orderBy(desc('total_units_sold')). \
    limit(5)
result.show()

+----------+----------------+
|      date|total_units_sold|
+----------+----------------+
|2023-07-07|         1104297|
|2023-07-01|         1102564|
|2023-07-04|         1102133|
|2023-07-02|         1101015|
|2023-07-10|         1100401|
+----------+----------------+



In [8]:
pandas_df = result.toPandas()
pandas_df.to_parquet('result.parquet')

In [9]:
spark.stop()